# YZTA Datathon 


In [1]:
!pip install lightgbm catboost optuna -q

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

## 1. Veri Yükleme

In [3]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test_x.csv')

train_id = train['id'].copy()
test_id  = test['id'].copy()
target   = train['bilissel_performans_skoru'].copy()

train = train.drop(columns=['id', 'bilissel_performans_skoru'])
test  = test.drop(columns=['id'])

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target — Min: {target.min():.2f} | Max: {target.max():.2f} | Ort: {target.mean():.2f} | Std: {target.std():.2f}')
print(f'Skewness: {target.skew():.4f}')  # log transform'a gerek var mı?

Train: (56000, 22) | Test: (24000, 22)
Target — Min: 0.00 | Max: 10.00 | Ort: 5.91 | Std: 2.23
Skewness: -0.2885


## 2. Hedef Değişken Analizi — Log Transform Kararı

Skewness |> 0.75 ise log transform faydalıdır, aksi halde direkt eğitiriz.

In [4]:
skewness = target.skew()

target_min = target.min()
SHIFT = max(0, -target_min + 0.01)  # negatif değer varsa kaydır

USE_LOG = abs(skewness) > 0.75
print(f'Skewness: {skewness:.4f} — Log Transform: {"AÇIK" if USE_LOG else "KAPALI"}')

if USE_LOG:
    target_transformed = np.log1p(target + SHIFT)
    print(f'Shift: {SHIFT:.4f} | Log target — Min: {target_transformed.min():.3f} | Max: {target_transformed.max():.3f}')
else:
    target_transformed = target.copy()
    SHIFT = 0
    print('Log transform uygulanmadı, orijinal target kullanılıyor.')

Skewness: -0.2885 — Log Transform: KAPALI
Log transform uygulanmadı, orijinal target kullanılıyor.


## 3. Metin Normalizasyonu ve Eksik Değer Dolumu

In [5]:
ulke_mapping = {
    'spain'      : 'ispanya',
    'south korea': 'guney kore',
    'sweden'     : 'isvec',
    'netherlands': 'hollanda',
    'mexico'     : 'meksika',
    'china'      : 'cin',
}
meslek_mapping = {'lawyer': 'avukat'}

cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Metin temizliği ve bilinmiyor doldurma
for col in cat_cols:
    train[col] = train[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()
    test[col]  = test[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()

train['ulke']   = train['ulke'].replace(ulke_mapping)
test['ulke']    = test['ulke'].replace(ulke_mapping)
train['meslek'] = train['meslek'].replace(meslek_mapping)
test['meslek']  = test['meslek'].replace(meslek_mapping)

# Sayısal eksikleri train medyanıyla doldur (test sızıntısını önlemek için)
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col]  = test[col].fillna(median_val)

# Aykırı değer traşlama — sınırlar daima train'den hesaplanır
for col in num_cols:
    lower = train[col].quantile(0.01)
    upper = train[col].quantile(0.99)
    train[col] = train[col].clip(lower, upper)
    test[col]  = test[col].clip(lower, upper)

# Kategori dağılımlarını kontrol et
for col in cat_cols:
    n = pd.concat([train[col], test[col]]).nunique()
    print(f'{col}: {n} benzersiz değer')

print('\nTemizlik tamam.')

cinsiyet: 2 benzersiz değer
meslek: 12 benzersiz değer
ulke: 12 benzersiz değer
kronotip: 4 benzersiz değer
ruh_sagligi_durumu: 5 benzersiz değer
mevsim: 2 benzersiz değer
gun_tipi: 2 benzersiz değer

Temizlik tamam.


## 4. Kategorik Encoding

**v2'deki kritik bug:** `mevsim` binary_cols içindeydi ama 4 mevsim var (ilkbahar/yaz/sonbahar/kış).
Düzeltme: binary olan sadece `cinsiyet` ve `gun_tipi`.
`mevsim` için ordinal da yanlış — mevsimler arasında büyüklük ilişkisi yok (ilkbahar < yaz gibi bir anlam taşımaz).
Doğru yol: `mevsim` de target encoding ile işlenir, tıpkı `kronotip` ve `ruh_sagligi_durumu` gibi.

In [6]:
# Sadece gerçekten 2 değerli olanlar binary encode edilir
binary_cols = ['cinsiyet', 'gun_tipi']

for col in binary_cols:
    categories = sorted(
        pd.concat([train[col], test[col]], ignore_index=True).unique()
    )
    mapping = {cat: i for i, cat in enumerate(categories)}
    train[col] = train[col].map(mapping)
    test[col]  = test[col].map(mapping)
    print(f'{col}: {mapping}')

print('\nBinary encoding tamam.')

cinsiyet: {'erkek': 0, 'kadin': 1}
gun_tipi: {'hafta ici': 0, 'hafta sonu': 1}

Binary encoding tamam.


In [7]:
# KFold Smoothed Target Encoding — nominal kategorikler için
# mevsim dahil: 4 mevsim arasında büyüklük sırası yok, target encoding doğru seçim
# SMOOTH=15: 10-20 bandında, az gözlemlenen kategorilerde global ortalamaya çeker
TARGET_ENCODE_COLS = ['kronotip', 'ruh_sagligi_durumu', 'meslek', 'ulke', 'mevsim']
N_FOLDS = 5
SMOOTH  = 15

global_mean = target_transformed.mean()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for col in TARGET_ENCODE_COLS:
    agg = pd.DataFrame({'col': train[col].values, 'target': target_transformed.values})

    # Test için: tüm train verisiyle smoothed ortalama
    stats = agg.groupby('col')['target'].agg(['mean', 'count'])
    smoothed_map = (
        (stats['count'] * stats['mean'] + SMOOTH * global_mean)
        / (stats['count'] + SMOOTH)
    ).to_dict()

    # Train için: OOF encoding (her satır kendi foldunu görmez)
    oof_encoded = np.full(len(train), global_mean, dtype=np.float64)

    for tr_idx, val_idx in kf.split(train):
        fold_agg   = agg.iloc[tr_idx]
        fold_stats = fold_agg.groupby('col')['target'].agg(['mean', 'count'])
        fold_map   = (
            (fold_stats['count'] * fold_stats['mean'] + SMOOTH * global_mean)
            / (fold_stats['count'] + SMOOTH)
        ).to_dict()

        val_col = train[col].iloc[val_idx]
        oof_encoded[val_idx] = val_col.map(fold_map).fillna(global_mean).values

    train[col] = oof_encoded
    test[col]  = test[col].map(smoothed_map).fillna(global_mean)

    print(f'  {col}: tamam')

print(f'\nEncoding bitti. Boyut — Train: {train.shape} | Test: {test.shape}')

  kronotip: tamam
  ruh_sagligi_durumu: tamam
  meslek: tamam
  ulke: tamam
  mevsim: tamam

Encoding bitti. Boyut — Train: (56000, 22) | Test: (24000, 22)


## 5. Özellik Mühendisliği

In [8]:
# Meslek × gün tipi kombinasyonu (encoding sonrası numerik çarpım)
train['meslek_gun_tipi'] = train['meslek'] * train['gun_tipi']
test['meslek_gun_tipi']  = test['meslek'] * test['gun_tipi']

# Uyku kalite endeksi: kaliteli uyku / bölünme sayısı
train['uyku_kalite_endeksi'] = (
    (train['rem_yuzdesi'] + train['derin_uyku_yuzdesi'])
    / (train['gecelik_uyanma_sayisi'] + 1)
)
test['uyku_kalite_endeksi'] = (
    (test['rem_yuzdesi'] + test['derin_uyku_yuzdesi'])
    / (test['gecelik_uyanma_sayisi'] + 1)
)

# Toplam kaliteli uyku yüzdesi
train['toplam_kaliteli_uyku'] = train['rem_yuzdesi'] + train['derin_uyku_yuzdesi']
test['toplam_kaliteli_uyku']  = test['rem_yuzdesi'] + test['derin_uyku_yuzdesi']

# Zihinsel yük: stres × çalışma saati
train['zihinsel_yuk'] = train['stres_skoru'] * train['gunluk_calisma_saati']
test['zihinsel_yuk']  = test['stres_skoru'] * test['gunluk_calisma_saati']

# Uyku bozulma skoru: uyanma × uykuya dalma süresi
train['uyku_bozulma_skoru'] = train['gecelik_uyanma_sayisi'] * train['uykuya_dalma_suresi_dk']
test['uyku_bozulma_skoru']  = test['gecelik_uyanma_sayisi'] * test['uykuya_dalma_suresi_dk']

# Kafein × ekran süresi: uyku sabotaj endeksi
train['ekran_kafein'] = train['uyku_oncesi_ekran_suresi_dk'] * train['uyku_oncesi_kafein_mg']
test['ekran_kafein']  = test['uyku_oncesi_ekran_suresi_dk'] * test['uyku_oncesi_kafein_mg']

# Stres / uyku kalitesi oranı
train['stres_uyku_orani'] = train['stres_skoru'] / (train['uyku_kalite_endeksi'] + 1)
test['stres_uyku_orani']  = test['stres_skoru'] / (test['uyku_kalite_endeksi'] + 1)

# Yaş × stres: yaşla birlikte stres etkisi değişir
train['yas_stres'] = train['yas'] * train['stres_skoru']
test['yas_stres']  = test['yas'] * test['stres_skoru']

print(f'Toplam özellik sayısı: {train.shape[1]}')

Toplam özellik sayısı: 30


## 6. Özellik Seçimi — Permutation Importance Tabanlı

Basit korelasyon yerine LightGBM'in feature importance'ını kullanıyoruz.
Bu, non-lineer ilişkileri de yakalar.

In [9]:
X_all = train.copy()
y_all = target_transformed.copy()

# Hızlı bir LGB ile feature importance al
selector = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    random_state=SEED, verbose=-1
)
selector.fit(X_all, y_all)

importance_df = pd.DataFrame({
    'ozellik'   : X_all.columns,
    'importance': selector.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature Importance (ilk 20):')
print(importance_df.head(20).to_string(index=False))

# Sıfır importance'lı özellikleri at (gerçekten işe yaramayan)
drop_features = importance_df[importance_df['importance'] == 0]['ozellik'].tolist()
print(f'\nAtılacak özellikler ({len(drop_features)}): {drop_features}')

train = train.drop(columns=drop_features, errors='ignore')
test  = test.drop(columns=drop_features, errors='ignore')
print(f'Kalan özellik sayısı: {train.shape[1]}')

Feature Importance (ilk 20):
                    ozellik  importance
                rem_yuzdesi         893
         gunluk_adim_sayisi         878
                stres_skoru         856
       toplam_kaliteli_uyku         825
      oda_sicakligi_celsius         794
                     meslek         729
         ruh_sagligi_durumu         695
         derin_uyku_yuzdesi         625
        vucut_kitle_indeksi         595
 hafta_sonu_uyku_farki_saat         577
     uykuya_dalma_suresi_dk         554
           stres_uyku_orani         546
uyku_oncesi_ekran_suresi_dk         510
                  yas_stres         507
        sekerleme_suresi_dk         487
        uyku_kalite_endeksi         481
               ekran_kafein         467
       gunluk_calisma_saati         460
               zihinsel_yuk         451
                       ulke         444

Atılacak özellikler (0): []
Kalan özellik sayısı: 30


## 7. Optuna ile Hyperparameter Tuning (LightGBM)

Yalnızca LGB için yapıyoruz çünkü en hızlı o. Bulunan parametreler XGB/CAT için de referans.

In [ ]:
X = train.copy()
y = target_transformed.copy()
kf_tune = KFold(n_splits=5, shuffle=True, random_state=SEED)

def lgb_objective(trial):
    params = {
        'objective'        : 'regression',
        'metric'           : 'rmse',
        'verbosity'        : -1,
        'random_state'     : SEED,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 80),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq'     : 5,
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.0, 2.0),
    }
    oof = np.zeros(len(X))
    for tr_idx, val_idx in kf_tune.split(X):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dval   = lgb.Dataset(X.iloc[val_idx], label=y.iloc[val_idx], reference=dtrain)
        model  = lgb.train(
            params, dtrain,
            num_boost_round=2000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
        )
        oof[val_idx] = model.predict(X.iloc[val_idx])
    return np.sqrt(mean_squared_error(y, oof))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

best_lgb_params = study.best_params
best_lgb_params.update({'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'random_state': SEED, 'bagging_freq': 5})

print(f'\nEn iyi LGB CV RMSE: {study.best_value:.5f}')
print(f'En iyi parametreler: {best_lgb_params}')

## 8. Model Eğitimi — 5-Fold CV ile 3'lü Ensemble

LightGBM + XGBoost + CatBoost → OOF tabanlı stacking (Ridge meta-model)

In [ ]:
N_FOLDS_MODEL = 5
kf_model = KFold(n_splits=N_FOLDS_MODEL, shuffle=True, random_state=SEED)

lgb_oof  = np.zeros(len(X))
xgb_oof  = np.zeros(len(X))
cat_oof  = np.zeros(len(X))

lgb_test = np.zeros(len(test))
xgb_test = np.zeros(len(test))
cat_test = np.zeros(len(test))

lgb_scores = []
xgb_scores = []
cat_scores = []

# XGB parametreleri — LGB tuning sonucuna yakın değerler
xgb_params = dict(
    n_estimators          = 3000,
    learning_rate         = best_lgb_params.get('learning_rate', 0.01),
    max_depth             = 6,
    min_child_weight      = best_lgb_params.get('min_child_samples', 40),
    subsample             = best_lgb_params.get('bagging_fraction', 0.75),
    colsample_bytree      = best_lgb_params.get('feature_fraction', 0.75),
    reg_alpha             = best_lgb_params.get('reg_alpha', 0.1),
    reg_lambda            = best_lgb_params.get('reg_lambda', 0.5),
    early_stopping_rounds = 150,
    eval_metric           = 'rmse',
    random_state          = SEED,
    verbosity             = 0,
)

print('=== 5-Fold Ensemble Eğitimi ===\n')

for fold, (tr_idx, val_idx) in enumerate(kf_model.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    print(f'--- Fold {fold + 1} ---')

    # ── LightGBM ────────────────────────────────────────────────────────────
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    lgb_model = lgb.train(
        best_lgb_params, dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )
    lgb_oof[val_idx] = lgb_model.predict(X_val)
    lgb_test        += lgb_model.predict(test) / N_FOLDS_MODEL
    lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_oof[val_idx]))
    lgb_scores.append(lgb_rmse)
    print(f'  LGB  RMSE: {lgb_rmse:.5f}')

    # ── XGBoost ─────────────────────────────────────────────────────────────
    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx] = xgb_model.predict(X_val)
    xgb_test        += xgb_model.predict(test) / N_FOLDS_MODEL
    xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_oof[val_idx]))
    xgb_scores.append(xgb_rmse)
    print(f'  XGB  RMSE: {xgb_rmse:.5f}')

    # ── CatBoost ────────────────────────────────────────────────────────────
    cat_model = CatBoostRegressor(
        iterations            = 3000,
        learning_rate         = best_lgb_params.get('learning_rate', 0.01),
        depth                 = 6,
        min_data_in_leaf      = best_lgb_params.get('min_child_samples', 40),
        subsample             = best_lgb_params.get('bagging_fraction', 0.75),
        l2_leaf_reg           = 3.0,
        early_stopping_rounds = 150,
        eval_metric           = 'RMSE',
        random_seed           = SEED,
        verbose               = False,
    )
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    cat_oof[val_idx] = cat_model.predict(X_val)
    cat_test        += cat_model.predict(test) / N_FOLDS_MODEL
    cat_rmse = np.sqrt(mean_squared_error(y_val, cat_oof[val_idx]))
    cat_scores.append(cat_rmse)
    print(f'  CAT  RMSE: {cat_rmse:.5f}\n')

lgb_oof_rmse = np.sqrt(mean_squared_error(y, lgb_oof))
xgb_oof_rmse = np.sqrt(mean_squared_error(y, xgb_oof))
cat_oof_rmse = np.sqrt(mean_squared_error(y, cat_oof))

print('=== CV Özeti ===')
print(f'LGB  OOF RMSE: {lgb_oof_rmse:.5f} (±{np.std(lgb_scores):.5f})')
print(f'XGB  OOF RMSE: {xgb_oof_rmse:.5f} (±{np.std(xgb_scores):.5f})')
print(f'CAT  OOF RMSE: {cat_oof_rmse:.5f} (±{np.std(cat_scores):.5f})')

In [ ]:
from sklearn.metrics import mean_squared_error

# Skorları saklamak için liste
rmse_scores = []

# İşte o meşhur döngü (Pistteki turlar)
for fold, (train_index, val_index) in enumerate(kf.split(X)):
    # 1. Veriyi parçalara ayır
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]
    
    # 2. Modeli eğit (Örn: XGBoost)
    model_xgb.fit(X_train_fold, y_train_fold)
    
    # 3. Bu parça için tahmin yap
    fold_preds = model_xgb.predict(X_val_fold)
    
    # 4. Bu turun hatasını hesapla (RMSE)
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, fold_preds))
    rmse_scores.append(fold_rmse)
    
    # 5. Ekrana Yazdır (İşte burada sonucu göreceksin!)
    print(f"Fold {fold+1} bitti. Skor: {fold_rmse:.4f}")

# Final sonucu
print(f"\n--- GENEL CV SONUCUN (Ortalama RMSE): {np.mean(rmse_scores):.4f} ---")

## 9. Stacking Ensemble — Ridge Meta-Model

Sabit ağırlık yerine Ridge regresyon meta-modeli kullanıyoruz.
Meta-model OOF tahminleri üzerinde fit ediliyor — test sızıntısı yok.

In [ ]:
# OOF tahminlerini meta-feature olarak bir araya getir
meta_train = np.column_stack([lgb_oof, xgb_oof, cat_oof])
meta_test  = np.column_stack([lgb_test, xgb_test, cat_test])

# Ridge meta-model — alpha=1.0 overfit'i bastırır
meta_model = Ridge(alpha=1.0, fit_intercept=True)
meta_model.fit(meta_train, y)

stacked_oof  = meta_model.predict(meta_train)
stacked_rmse = np.sqrt(mean_squared_error(y, stacked_oof))

print(f'Stacking OOF RMSE: {stacked_rmse:.5f}')
print(f'Meta-model ağırlıkları — LGB: {meta_model.coef_[0]:.4f} | XGB: {meta_model.coef_[1]:.4f} | CAT: {meta_model.coef_[2]:.4f}')

# Test tahminleri
test_preds_log = meta_model.predict(meta_test)

## 10. Log Dönüşümünü Geri Al ve Submission

In [ ]:
if USE_LOG:
    test_preds_final = np.expm1(test_preds_log) - SHIFT
else:
    test_preds_final = test_preds_log

# Gözlenen değer sınırlarına kliple (log geri dönüşüm taşabilir)
test_preds_final = np.clip(test_preds_final, target.min(), target.max())

print('Tahmin istatistikleri:')
print(f'  Min : {test_preds_final.min():.4f}')
print(f'  Max : {test_preds_final.max():.4f}')
print(f'  Ort : {test_preds_final.mean():.4f}')
print(f'  Std : {test_preds_final.std():.4f}')

submission = pd.DataFrame({
    'id'                       : test_id,
    'bilissel_performans_skoru': test_preds_final,
})

submission.to_csv('submission.csv', index=False)
print(f'\nsubmission.csv kaydedildi — {len(submission)} satır')

In [ ]:
from IPython.display import FileLink
FileLink('submission.csv')